In [2]:
import Pkg
Pkg.activate("..")
Pkg.status()

  Activating project at `~/quantum_computing/quantum_stochastic_programming`


Status `~/quantum_computing/quantum_stochastic_programming/Project.toml`
  [07493b3f] Alpine v0.5.8
  [336ed68f] CSV v0.10.16
  [a93c6f00] DataFrames v1.8.2
  [bb8be931] EAGO v0.9.2
  [87dc4568] HiGHS v1.23.0
  [b6b21f68] Ipopt v1.15.0
  [4076af6c] JuMP v1.30.1
  [2ddba703] Juniper v0.9.4
  [19f71287] MAiNGO v0.2.2
  [82193955] SCIP v0.12.8


In [3]:
import JuMP

# import Alpine
import EAGO
import HiGHS
import Ipopt
import Juniper
# import MAiNGO

In [4]:
import Random

In [5]:
d = 8
# frac = 0.5

# ps = [0.1, 0.9]
# @assert(sum(ps) == 1.0)
# ns = length(ps)

# ws_max = [
#     0.0 1.0;
#     1.0 1.0;
#     0.0 1.0;
# ]

cx = [
    4.0 1e-2;
    # 5.0 1e-2;
]
nx = size(cx, 1)
xlb = zeros(nx)
# xub = fill(5.0, nx)
xub = fill(10.0, nx)

# cw = [
#     0.25 1e-3;
#     0.3  1e-3;
#     0.35 1e-3;
# ]
# nw = size(cw,1)
ny = 4
# cy1 = 0.25 .+ 0.0:0.5:0.5*nw
cy1 = range(0.1, 1.0, ny)
cy2 = fill(1e-3, ny)
cy = hcat(cy1, cy2)

cr = 10.0

ns = 2^ny
ps = ones(ns) ./ ns
@assert(isapprox(sum(ps), 1.0))

# rng = Random.MersenneTwister(1234)
# xi = Random.rand(rng, ny, ns) .>= frac

xi = [((s-1) >> (ny-i)) & 1 for i in 1:ny, s in 1:ns]
# xi = zeros(ny, ns)
# idx = 0
# for k in 0:ny
#     @show binomial(ny, k)
#     for i in 1:binomial(ny, k)
#         idx += 1

#     end
# end
# @show idx

4×16 Matrix{Int64}:
 0  0  0  0  0  0  0  0  1  1  1  1  1  1  1  1
 0  0  0  0  1  1  1  1  0  0  0  0  1  1  1  1
 0  0  1  1  0  0  1  1  0  0  1  1  0  0  1  1
 0  1  0  1  0  1  0  1  0  1  0  1  0  1  0  1

In [78]:
#### Alpine ####
# ipopt = JuMP.optimizer_with_attributes(Ipopt.Optimizer, "print_level" => 0)
# highs = JuMP.optimizer_with_attributes(HiGHS.Optimizer, "output_flag" => false)
# solver = JuMP.optimizer_with_attributes(Alpine.Optimizer, "nlp_solver" => ipopt, "mip_solver" => highs)
#### EAGO ####
# solver = EAGO.Optimizer
#### HiGHS ####
# solver = HiGHS.Optimizer
#### Ipopt ####
solver = Ipopt.Optimizer
#### Juniper ####
# ipopt = JuMP.optimizer_with_attributes(Ipopt.Optimizer, "print_level"=>0)
# solver = JuMP.optimizer_with_attributes(Juniper.Optimizer, "nl_solver"=>ipopt)
#### MAiNGO ####
# solver = JuMP.optimizer_with_attributes(MAiNGO.Optimizer, "epsilonA"=> 1e-8)
m = JuMP.Model(solver)
# JuMP.@variable(m, x[1:nx] >= 0)
JuMP.@variable(m, xlb[i] <= x[i=1:nx] <= xub[i])
JuMP.@variable(m, y[i=1:ny, s=1:ns], lower_bound=0.0, upper_bound=1.0)
# JuMP.@variable(m, y[i=1:ny, s=1:ns], lower_bound=0.0, upper_bound=1.0, binary=true)

4×16 Matrix{JuMP.VariableRef}:
 y[1,1]  y[1,2]  y[1,3]  y[1,4]  y[1,5]  y[1,6]  …  y[1,12]  y[1,13]  y[1,14]  y[1,15]  y[1,16]
 y[2,1]  y[2,2]  y[2,3]  y[2,4]  y[2,5]  y[2,6]     y[2,12]  y[2,13]  y[2,14]  y[2,15]  y[2,16]
 y[3,1]  y[3,2]  y[3,3]  y[3,4]  y[3,5]  y[3,6]     y[3,12]  y[3,13]  y[3,14]  y[3,15]  y[3,16]
 y[4,1]  y[4,2]  y[4,3]  y[4,4]  y[4,5]  y[4,6]     y[4,12]  y[4,13]  y[4,14]  y[4,15]  y[4,16]

In [79]:
obj_fs = JuMP.@expression(
    m, obj_first_stage,
    sum(cx[i, 1] * x[i] + cx[i, 2] * x[i] * x[i] for i in 1:nx)
)

obj_fs_grad = JuMP.@expression(
    m, obj_first_stage_grad,
    sum(cx[i, 1] + cx[i, 2] * x[i] for i in 1:nx)
)

obj_ss = JuMP.@expression(
    m, obj_second_stage,
    sum(ps[s] * (cy[i, 1]*y[i, s] + cy[i, 2]*y[i, s]^2) * xi[i, s] for i in 1:ny, s in 1:ns)
    +
    sum(ps[s] * (cr * y[i, s]) * (1 - xi[i, s]) for i in 1:ny, s in 1:ns)
)

obj_ss_grad = JuMP.@expression(
    m, obj_second_stage_grad,
    1.0 / ny * sum(ps[s] * (cy[i, 1] + cy[i, 2]*y[i, s]) * xi[i, s] for i in 1:ny, s in 1:ns)
    +
    1.0 / ny * sum(ps[s] * cr * (1 - xi[i, s]) for i in 1:ny, s in 1:ns)
)

JuMP.@objective(m, Min, obj_fs + obj_ss)

# JuMP.@objective(m, Min, 
#     sum(cx[i,1] * x[i] + cx[i,2] * x[i] * x[i] for i in 1:nx)
#     + sum(ps[s] * (cy[i,1]*y[i,s] + cy[i,2]*y[i,s]^2) * xi[i,s] for i in 1:ny, s in 1:ns)
#     + sum(ps[s] * (cr * y[i,s]) * (1 - xi[i,s]) for i in 1:ny, s in 1:ns)
# )
# JuMP.@objective(m, Min, 
#     sum(cx[i,1] * x[i] for i in 1:nx)
#     + sum(ps[s] * cy[i,1] * y[i,s] * xi[i,s] for i in 1:ny, s in 1:ns)
#     + sum(ps[s] * cr * y[i,s] * (1 - xi[i,s]) for i in 1:ny, s in 1:ns)
# )

0.01 x[1]² + 6.25e-5 y[1,9]² + 6.25e-5 y[1,10]² + 6.25e-5 y[1,11]² + 6.25e-5 y[1,12]² + 6.25e-5 y[1,13]² + 6.25e-5 y[1,14]² + 6.25e-5 y[1,15]² + 6.25e-5 y[1,16]² + 6.25e-5 y[2,5]² + 6.25e-5 y[2,6]² + 6.25e-5 y[2,7]² + 6.25e-5 y[2,8]² + 6.25e-5 y[2,13]² + 6.25e-5 y[2,14]² + 6.25e-5 y[2,15]² + 6.25e-5 y[2,16]² + 6.25e-5 y[3,3]² + 6.25e-5 y[3,4]² + 6.25e-5 y[3,7]² + 6.25e-5 y[3,8]² + 6.25e-5 y[3,11]² + 6.25e-5 y[3,12]² + 6.25e-5 y[3,15]² + 6.25e-5 y[3,16]² + 6.25e-5 y[4,2]² + 6.25e-5 y[4,4]² + 6.25e-5 y[4,6]² + 6.25e-5 y[4,8]² + 6.25e-5 y[4,10]² + 6.25e-5 y[4,12]² + 6.25e-5 y[4,14]² + 6.25e-5 y[4,16]² + 4 x[1] + 0.00625 y[1,9] + 0.00625 y[1,10] + 0.00625 y[1,11] + 0.00625 y[1,12] + 0.00625 y[1,13] + 0.00625 y[1,14] + 0.00625 y[1,15] + 0.00625 y[1,16] + 0.025 y[2,5] + 0.025 y[2,6] + 0.025 y[2,7] + 0.025 y[2,8] + 0.025 y[2,13] + 0.025 y[2,14] + 0.025 y[2,15] + 0.025 y[2,16] + 0.04375 y[3,3] + 0.04375 y[3,4] + 0.04375 y[3,7] + 0.04375 y[3,8] + 0.04375 y[3,11] + 0.04375 y[3,12] + 0.04375 y[3,

In [80]:
for s in 1:ns
    JuMP.@constraint(m, sum(x[i] for i in 1:nx) + sum(y[i,s] for i in 1:ny) == d)
end

In [81]:
display(m)

A JuMP Model
├ solver: Ipopt
├ objective_sense: MIN_SENSE
│ └ objective_function_type: JuMP.QuadExpr
├ num_variables: 65
├ num_constraints: 146
│ ├ JuMP.AffExpr in MOI.EqualTo{Float64}: 16
│ ├ JuMP.VariableRef in MOI.GreaterThan{Float64}: 65
│ └ JuMP.VariableRef in MOI.LessThan{Float64}: 65
└ Names registered in the model
  └ :obj_first_stage, :obj_first_stage_grad, :obj_second_stage, :obj_second_stage_grad, :x, :y

In [82]:
JuMP.all_constraints(m, include_variable_in_set_constraints=true)

146-element Vector{JuMP.ConstraintRef}:
 x[1] + y[1,1] + y[2,1] + y[3,1] + y[4,1] = 8
 x[1] + y[1,2] + y[2,2] + y[3,2] + y[4,2] = 8
 x[1] + y[1,3] + y[2,3] + y[3,3] + y[4,3] = 8
 x[1] + y[1,4] + y[2,4] + y[3,4] + y[4,4] = 8
 x[1] + y[1,5] + y[2,5] + y[3,5] + y[4,5] = 8
 x[1] + y[1,6] + y[2,6] + y[3,6] + y[4,6] = 8
 x[1] + y[1,7] + y[2,7] + y[3,7] + y[4,7] = 8
 x[1] + y[1,8] + y[2,8] + y[3,8] + y[4,8] = 8
 x[1] + y[1,9] + y[2,9] + y[3,9] + y[4,9] = 8
 x[1] + y[1,10] + y[2,10] + y[3,10] + y[4,10] = 8
 x[1] + y[1,11] + y[2,11] + y[3,11] + y[4,11] = 8
 x[1] + y[1,12] + y[2,12] + y[3,12] + y[4,12] = 8
 x[1] + y[1,13] + y[2,13] + y[3,13] + y[4,13] = 8
 ⋮
 y[1,14] ≤ 1
 y[2,14] ≤ 1
 y[3,14] ≤ 1
 y[4,14] ≤ 1
 y[1,15] ≤ 1
 y[2,15] ≤ 1
 y[3,15] ≤ 1
 y[4,15] ≤ 1
 y[1,16] ≤ 1
 y[2,16] ≤ 1
 y[3,16] ≤ 1
 y[4,16] ≤ 1

In [83]:
JuMP.optimize!(m)

This is Ipopt version 3.14.19, running with linear solver MUMPS 5.8.1.

Number of nonzeros in equality constraint Jacobian...:       80
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:       33

Total number of variables............................:       65
                     variables with only lower bounds:        0
                variables with lower and upper bounds:       65
                     variables with only upper bounds:        0
Total number of equality constraints.................:       16
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  2.5100095e-01 7.95e+00 5.14e-01  -1.0 0.00e+00    -  0.00e+00 0.00e+00 

In [84]:
JuMP.solution_summary(m)

solution_summary(; result = 1, verbose = false)
├ solver_name          : Ipopt
├ Termination
│ ├ termination_status : LOCALLY_SOLVED
│ ├ result_count       : 1
│ └ raw_status         : Solve_Succeeded
├ Solution (result = 1)
│ ├ primal_status        : FEASIBLE_POINT
│ ├ dual_status          : FEASIBLE_POINT
│ ├ objective_value      : 2.88741e+01
│ └ dual_objective_value : 2.92357e+01
└ Work counters
  ├ solve_time (sec)   : 3.36099e-03
  └ barrier_iterations : 9

In [85]:
JuMP.value(x)

1-element Vector{Float64}:
 6.000000046812948

In [86]:
JuMP.value(y)

4×16 Matrix{Float64}:
 0.5  0.333333  0.333333  -4.76607e-9  0.333333  …   1.0          1.0         1.0
 0.5  0.333333  0.333333  -4.76607e-9  1.0           1.0          1.0         1.0
 0.5  0.333333  1.0        1.0         0.333333     -5.66662e-9   2.95916e-7  2.59875e-7
 0.5  1.0       0.333333   1.0         0.333333      1.46179e-7  -5.73659e-9  8.03231e-8

In [87]:
idx = abs.(JuMP.value(y) .- Int.(round.(JuMP.value(y)))) .> 1e-6
println("Number of fractional values: ", sum(idx))
;

Number of fractional values: 16


In [88]:
xi

4×16 Matrix{Int64}:
 0  0  0  0  0  0  0  0  1  1  1  1  1  1  1  1
 0  0  0  0  1  1  1  1  0  0  0  0  1  1  1  1
 0  0  1  1  0  0  1  1  0  0  1  1  0  0  1  1
 0  1  0  1  0  1  0  1  0  1  0  1  0  1  0  1

In [89]:
report = JuMP.primal_feasibility_report(m)
max_viol = isempty(report) ? 0.0 : maximum(values(report))

5.947780046255957e-9

In [90]:
@show JuMP.objective_value(m)
# @show sum(cx[:,1] .* JuMP.value(x))
# @show JuMP.objective_value(m) - sum(cx[:,1] .* JuMP.value(x))
@show sum(cx[:,1] .* JuMP.value(x) + cx[:,2] .* JuMP.value(x).^2)
@show JuMP.objective_value(m) - sum(cx[:,1] .* JuMP.value(x) + cx[:,2] .* JuMP.value(x).^2)
;

JuMP.objective_value(m) = 28.874125003433765
sum(cx[:, 1] .* JuMP.value(x) + cx[:, 2] .* JuMP.value(x) .^ 2) = 24.360000192869347
JuMP.objective_value(m) - sum(cx[:, 1] .* JuMP.value(x) + cx[:, 2] .* JuMP.value(x) .^ 2) = 4.514124810564418


In [91]:
nres = (xi - JuMP.value(y)) .< 0
@show sum(nres)
@show sum(nres) / (ny * ns)
;

sum(nres) = 20
sum(nres) / (ny * ns) = 0.3125


In [92]:
@show m[:obj_first_stage]
JuMP.value(m[:obj_first_stage])

m[:obj_first_stage] = 0.01 x[1]² + 4 x[1]


24.360000192869347

In [93]:
@show m[:obj_second_stage]
JuMP.value(m[:obj_second_stage])

m[:obj_second_stage] = 6.25e-5 y[1,9]² + 6.25e-5 y[1,10]² + 6.25e-5 y[1,11]² + 6.25e-5 y[1,12]² + 6.25e-5 y[1,13]² + 6.25e-5 y[1,14]² + 6.25e-5 y[1,15]² + 6.25e-5 y[1,16]² + 6.25e-5 y[2,5]² + 6.25e-5 y[2,6]² + 6.25e-5 y[2,7]² + 6.25e-5 y[2,8]² + 6.25e-5 y[2,13]² + 6.25e-5 y[2,14]² + 6.25e-5 y[2,15]² + 6.25e-5 y[2,16]² + 6.25e-5 y[3,3]² + 6.25e-5 y[3,4]² + 6.25e-5 y[3,7]² + 6.25e-5 y[3,8]² + 6.25e-5 y[3,11]² + 6.25e-5 y[3,12]² + 6.25e-5 y[3,15]² + 6.25e-5 y[3,16]² + 6.25e-5 y[4,2]² + 6.25e-5 y[4,4]² + 6.25e-5 y[4,6]² + 6.25e-5 y[4,8]² + 6.25e-5 y[4,10]² + 6.25e-5 y[4,12]² + 6.25e-5 y[4,14]² + 6.25e-5 y[4,16]² + 0.00625 y[1,9] + 0.00625 y[1,10] + 0.00625 y[1,11] + 0.00625 y[1,12] + 0.00625 y[1,13] + 0.00625 y[1,14] + 0.00625 y[1,15] + 0.00625 y[1,16] + 0.025 y[2,5] + 0.025 y[2,6] + 0.025 y[2,7] + 0.025 y[2,8] + 0.025 y[2,13] + 0.025 y[2,14] + 0.025 y[2,15] + 0.025 y[2,16] + 0.04375 y[3,3] + 0.04375 y[3,4] + 0.04375 y[3,7] + 0.04375 y[3,8] + 0.04375 y[3,11] + 0.04375 y[3,12] + 0.04375 y[3

4.514124810564424

In [94]:
@show m[:obj_first_stage_grad]
JuMP.value(m[:obj_first_stage_grad])

m[:obj_first_stage_grad] = 0.01 x[1] + 4


4.060000000468129

In [95]:
@show m[:obj_second_stage_grad]
JuMP.value(m[:obj_second_stage_grad])

m[:obj_second_stage_grad] = 1.5625e-5 y[1,9] + 1.5625e-5 y[1,10] + 1.5625e-5 y[1,11] + 1.5625e-5 y[1,12] + 1.5625e-5 y[1,13] + 1.5625e-5 y[1,14] + 1.5625e-5 y[1,15] + 1.5625e-5 y[1,16] + 1.5625e-5 y[2,5] + 1.5625e-5 y[2,6] + 1.5625e-5 y[2,7] + 1.5625e-5 y[2,8] + 1.5625e-5 y[2,13] + 1.5625e-5 y[2,14] + 1.5625e-5 y[2,15] + 1.5625e-5 y[2,16] + 1.5625e-5 y[3,3] + 1.5625e-5 y[3,4] + 1.5625e-5 y[3,7] + 1.5625e-5 y[3,8] + 1.5625e-5 y[3,11] + 1.5625e-5 y[3,12] + 1.5625e-5 y[3,15] + 1.5625e-5 y[3,16] + 1.5625e-5 y[4,2] + 1.5625e-5 y[4,4] + 1.5625e-5 y[4,6] + 1.5625e-5 y[4,8] + 1.5625e-5 y[4,10] + 1.5625e-5 y[4,12] + 1.5625e-5 y[4,14] + 1.5625e-5 y[4,16] + 5.275


5.275406249993591

In [98]:
for c in JuMP.all_constraints(m, include_variable_in_set_constraints=false)
    println(c)
    println(JuMP.dual(c))
end

x[1] + y[1,1] + y[2,1] + y[3,1] + y[4,1] = 8
0.6250000002326931
x[1] + y[1,2] + y[2,2] + y[3,2] + y[4,2] = 8
0.624999996596493
x[1] + y[1,3] + y[2,3] + y[3,3] + y[4,3] = 8
0.6249999965974655
x[1] + y[1,4] + y[2,4] + y[3,4] + y[4,4] = 8
0.14624247851020697
x[1] + y[1,5] + y[2,5] + y[3,5] + y[4,5] = 8
0.6249999965984018
x[1] + y[1,6] + y[2,6] + y[3,6] + y[4,6] = 8
0.1393403035761838
x[1] + y[1,7] + y[2,7] + y[3,7] + y[4,7] = 8
0.12801682250719984
x[1] + y[1,8] + y[2,8] + y[3,8] + y[4,8] = 8
0.05401528907212677
x[1] + y[1,9] + y[2,9] + y[3,9] + y[4,9] = 8
0.624999996599306
x[1] + y[1,10] + y[2,10] + y[3,10] + y[4,10] = 8
0.1336208191494644
x[1] + y[1,11] + y[2,11] + y[3,11] + y[4,11] = 8
0.12101032964861318
x[1] + y[1,12] + y[2,12] + y[3,12] + y[4,12] = 8
0.05248869792063589
x[1] + y[1,13] + y[2,13] + y[3,13] + y[4,13] = 8
0.10956322263513425
x[1] + y[1,14] + y[2,14] + y[3,14] + y[4,14] = 8
0.044569506219323235
x[1] + y[1,15] + y[2,15] + y[3,15] + y[4,15] = 8
0.035217621144289904
x[1] + y

In [104]:
sum(ps .* JuMP.dual.(JuMP.all_constraints(m, include_variable_in_set_constraints=false)))

0.25750000007092966

In [107]:
JuMP.dual.(JuMP.all_constraints(m, include_variable_in_set_constraints=true))[ns+1:end]

130-element Vector{Float64}:
  0.0
  0.0
  0.0
  0.0
  0.0
  3.4035070642863665e-9
  3.4035070642863665e-9
  3.4035070642863665e-9
  0.0
  3.4025345270322885e-9
  3.4025345270314415e-9
  0.0
  3.4025345270322885e-9
  ⋮
 -0.03819450622664534
 -0.019444506235416808
  0.0
  0.0
 -0.028842621154627674
 -0.010092621176076205
  0.0
  0.0
 -0.02453992413891154
 -0.0057899241641378866
  0.0
  0.0